# 05 — PROJECT: Custom CNN Classifier
**Phase 3 · Week 9 · Friday · Jun 5 2026 · Remote**

## Project Goal
Build a **complete end-to-end CNN training pipeline** from data loading to trained model:
- Custom `Dataset` class with transforms
- Data augmentation pipeline
- Full training + validation loop
- Best-model checkpoint saving and loading
- Accuracy/loss curves
- Confusion matrix analysis

This is the template you will reuse for every classification project in Phase 3 and beyond.

---

## Pipeline Overview
```
Raw Images
    ↓
Custom Dataset (ImageFolder or manual)
    ↓
Transforms: Resize → Augment → Normalize
    ↓
DataLoader (batch, shuffle, workers)
    ↓
CNN Model (conv blocks + classifier head)
    ↓
Training Loop: forward → loss → backward → step
    ↓
Validation Loop: eval mode, no_grad
    ↓
Checkpoint: save best val_acc model
    ↓
Analysis: confusion matrix, per-class accuracy
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Custom Dataset class ────────────────────────────────────────────────────
class SyntheticImageDataset(Dataset):
    """
    Drop-in template for a real image dataset.
    Replace __init__ with actual image loading, __getitem__ with PIL.Image.open().
    For real data use: torchvision.datasets.ImageFolder('path/to/data')
    """
    def __init__(self, n_samples=4000, n_classes=5, img_size=32, transform=None):
        self.transform = transform
        self.n_classes = n_classes
        # Synthetic: each class has a different mean colour pattern
        self.data, self.labels = [], []
        per_class = n_samples // n_classes
        for cls in range(n_classes):
            for _ in range(per_class):
                # Create 3-channel image with class-specific pattern
                img = np.random.randn(3, img_size, img_size).astype(np.float32)
                img[cls % 3] += 1.5       # each class brightens a different channel
                self.data.append(img)
                self.labels.append(cls)
        self.data   = np.stack(self.data)
        self.labels = np.array(self.labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.tensor(self.data[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

CLASS_NAMES = ['Cat', 'Dog', 'Bird', 'Car', 'Flower']

dataset = SyntheticImageDataset(n_samples=4000, n_classes=5)
print(f"Dataset size : {len(dataset)} samples")
print(f"Image shape  : {dataset[0][0].shape}")
print(f"Classes      : {CLASS_NAMES}")

---

## Data Augmentation Pipeline

Augmentation creates artificial variety, preventing overfitting and teaching the model invariance to real-world transformations.

| Transform | Purpose |
|---|---|
| `RandomHorizontalFlip` | Objects look the same flipped |
| `RandomCrop` | Invariant to small translations |
| `ColorJitter` | Robust to lighting changes |
| `RandomRotation` | Rotation invariance |
| `Normalize(mean, std)` | Zero-centre each channel for stable training |

**Rule:** Apply augmentation **only to training set**. Validation/test only gets Normalize.

In [ ]:
# ── Augmentation transforms ────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)], p=0.5),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

val_transform = transforms.Compose([
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

# ── Split into train/val ────────────────────────────────────────────────────
full_ds  = SyntheticImageDataset(n_samples=5000, n_classes=5)
n_val    = int(0.2 * len(full_ds))
n_train  = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

# Apply different transforms to each split
class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self):   return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform: img = self.transform(img)
        return img, label

train_dataset = TransformedSubset(train_ds, train_transform)
val_dataset   = TransformedSubset(val_ds,   val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} samples  ({len(train_loader)} batches)")
print(f"Val  : {len(val_dataset)}  samples  ({len(val_loader)}  batches)")

# Visualise one batch
Xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, img, lbl in zip(axes, Xb[:8], yb[:8]):
    img_show = (img.permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1)
    ax.imshow(img_show); ax.set_title(CLASS_NAMES[lbl], fontsize=7); ax.axis('off')
plt.suptitle("Training Batch Sample", fontsize=10)
plt.tight_layout(); plt.savefig("training_batch.png", dpi=120, bbox_inches='tight'); plt.show()

---

## CNN Model + Training Loop

Using the ConvNet from yesterday, adapted for 3-channel (RGB) input.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3×32×32 → 32×16×16
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2: 32×16×16 → 64×8×8
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3: 64×8×8 → 128×4×4
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)          # → 128×1×1
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

model = CustomCNN(num_classes=5).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"CustomCNN — {total:,} parameters")

# ── Training loop with checkpoint saving ───────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, n = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(torch.long).to(device)
        optimizer.zero_grad()
        out  = model(Xb)
        loss = criterion(out, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * Xb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        n          += Xb.size(0)
    return total_loss/n, correct/n

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(torch.long).to(device)
            out  = model(Xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * Xb.size(0)
            correct    += (out.argmax(1) == yb).sum().item()
            n          += Xb.size(0)
    return total_loss/n, correct/n

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
criterion = nn.CrossEntropyLoss()

history = defaultdict(list)
best_val_acc = 0.0
EPOCHS = 25

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()

    history['tr_loss'].append(tr_loss); history['tr_acc'].append(tr_acc)
    history['vl_loss'].append(vl_loss); history['vl_acc'].append(vl_acc)

    # Save best checkpoint
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_acc': vl_acc, 'optimizer_state': optimizer.state_dict()},
                   'best_cnn_checkpoint.pt')

    if epoch % 5 == 0 or epoch == EPOCHS-1:
        print(f"Epoch {epoch+1:2d} | tr {tr_loss:.4f}/{tr_acc:.3f} "
              f"| val {vl_loss:.4f}/{vl_acc:.3f} | best {best_val_acc:.3f}")

print(f"\nBest val accuracy: {best_val_acc:.3f}")

In [ ]:
# ── Loss & accuracy curves ─────────────────────────────────────────────────
epochs_ax = range(1, EPOCHS+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_ax, history['tr_loss'], label='Train', lw=2)
ax1.plot(epochs_ax, history['vl_loss'], label='Val',   lw=2, ls='--')
ax1.set_title("Loss Curves"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ax, history['tr_acc'], label='Train', lw=2)
ax2.plot(epochs_ax, history['vl_acc'], label='Val',   lw=2, ls='--')
ax2.axhline(best_val_acc, color='gold', ls=':', lw=1.5, label=f'Best {best_val_acc:.3f}')
ax2.set_title("Accuracy Curves"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("cnn_classifier_curves.png", dpi=120, bbox_inches='tight')
plt.show()

# ── Confusion matrix ────────────────────────────────────────────────────────
# Load best checkpoint and evaluate
ckpt = torch.load('best_cnn_checkpoint.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']+1} (val_acc={ckpt['val_acc']:.3f})")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for Xb, yb in val_loader:
        preds = model(Xb.to(device)).argmax(1).cpu()
        all_preds.append(preds); all_labels.append(yb)
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Build confusion matrix manually
n_cls = 5
cm = np.zeros((n_cls, n_cls), dtype=int)
for true, pred in zip(all_labels, all_preds):
    cm[true, pred] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n_cls)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(n_cls)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Best Checkpoint", fontweight='bold')
for i in range(n_cls):
    for j in range(n_cls):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=11)
plt.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches='tight')
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for cls in range(n_cls):
    mask = all_labels == cls
    acc  = (all_preds[mask] == cls).mean()
    print(f"  {CLASS_NAMES[cls]:8s}: {acc:.3f}  ({mask.sum()} samples)")

---

## What I Learned Today

**Custom Dataset** — Inherit `Dataset`, implement `__len__` and `__getitem__`. For real image data, use `torchvision.datasets.ImageFolder('data/')` which auto-assigns class labels from subfolder names.

**Augmentation** — Only applied to training data. `RandomHorizontalFlip`, `RandomCrop`, `ColorJitter` are the most useful for natural images. Augmentation is a free regularizer — always use it.

**Training loop pattern** — 4 steps every batch: `optimizer.zero_grad()` → `model(X)` → `loss.backward()` → `optimizer.step()`. Never forget `zero_grad()` — gradients accumulate by default.

**Checkpoint saving** — Save `model.state_dict()`, `optimizer.state_dict()`, and the epoch/metric. Load with `model.load_state_dict(ckpt['model_state'])`. Always save the BEST model, not the last epoch.

**Confusion matrix** — Shows which classes get confused with each other. Off-diagonal entries = mistakes. If Cat↔Dog is high, consider collecting more distinctive samples or stronger augmentation.

**Cosine Annealing** — Gradually reduces lr from initial value to near zero following a cosine curve. Avoids sudden drops and lets the model settle smoothly into a minimum.